# 03.1 — QAT Test-Set Inference using Pytorch

Evaluate QAT checkpoints on the **test set** (ImageNet val split).

- **INT8**: `checkpoints/qat/int8_in{8,4,2,1}b/`
- **INT4**: `checkpoints/qat/int4_in{8,4,2,1}b/`

Only runs for which training has completed (checkpoint exists) are evaluated.

Results saved to `resultsv2/test_runs/`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────
SPLIT = "val"
TEST_IMAGENET = "/home/pf4636/imagenet2"
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/qat_test"
SKIP_EXISTING = True

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [3]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from src.eval import evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [4]:
QAT_CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QAT_CONFIGS = [
    {"qat_precision": "int8", "input_bits": [8, 4, 2, 1]},
    {"qat_precision": "int4", "input_bits": [8, 4, 2, 1]},
]

checkpoints = {}
skipped = []
for qcfg in QAT_CONFIGS:
    prec = qcfg["qat_precision"]
    for bits in qcfg["input_bits"]:
        run_name = f"{prec}_in{bits}b"
        ckpt_dir = QAT_CKPT_ROOT / run_name
        fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / "seed_42" / "best.pth"
        weights = ckpt_dir / "qat_modelopt_best.pth"
        mostate = ckpt_dir / "qat_modelopt_best_mostate.pt"

        if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
            skipped.append(f"{prec}/in{bits}b")
            continue

        checkpoints[(prec, bits)] = {
            "weights": weights,
            "mostate": mostate,
            "fp32_ckpt": fp32_ckpt,
        }

print("QAT checkpoints found:")
for (prec, bits), paths in checkpoints.items():
    print(f"  {prec} / in{bits}b: {paths['weights'].parent.name}/{paths['weights'].name}")
if skipped:
    print(f"\nSkipped (training not complete): {', '.join(skipped)}")
print(f"Test set: {TEST_IMAGENET}")

QAT checkpoints found:
  int8 / in8b: int8_in8b/qat_modelopt_best.pth
  int8 / in4b: int8_in4b/qat_modelopt_best.pth
  int8 / in2b: int8_in2b/qat_modelopt_best.pth
  int8 / in1b: int8_in1b/qat_modelopt_best.pth
  int4 / in8b: int4_in8b/qat_modelopt_best.pth
  int4 / in4b: int4_in4b/qat_modelopt_best.pth
  int4 / in2b: int4_in2b/qat_modelopt_best.pth
  int4 / in1b: int4_in1b/qat_modelopt_best.pth
Test set: /home/pf4636/imagenet2


In [5]:
def load_qat_model(prec: str, bits: int) -> nn.Module:
    paths = checkpoints[(prec, bits)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_IMAGENET)
    dataset = build_imagenet_dataset(test_cfg, split=SPLIT)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## QAT INT8 — Test-Set Evaluation

In [6]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

int8_records = []
criterion = nn.CrossEntropyLoss()
int8_bits = [b for b in [8, 4, 2, 1] if ("int8", b) in checkpoints]

for bits in int8_bits:
    run_id = f"resnet18_qat_int8_in{bits}b_cuda_bs1"
    result_path = OUT_DIR / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  in{bits}b: exists, skipping")
        with open(result_path) as f:
            int8_records.append(json.load(f))
        continue

    print(f"\n--- QAT INT8, input_bits={bits} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model("int8", bits)
    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, test_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {"qat_precision": "int8", "input_quant_bits": bits, "seed": 42},
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    save_path = OUT_DIR / run_id / "result.json"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    int8_records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(int8_records)} INT8 runs complete.")

  in8b: exists, skipping
  in4b: exists, skipping
  in2b: exists, skipping
  in1b: exists, skipping

4 INT8 runs complete.


## QAT INT4 — Test-Set Evaluation

In [7]:
int4_records = []
int4_bits = [b for b in [8, 4, 2, 1] if ("int4", b) in checkpoints]

for bits in int4_bits:
    run_id = f"resnet18_qat_int4_in{bits}b_cuda_bs1"
    result_path = OUT_DIR / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  in{bits}b: exists, skipping")
        with open(result_path) as f:
            int4_records.append(json.load(f))
        continue

    print(f"\n--- QAT INT4, input_bits={bits} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model("int4", bits)
    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, test_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {"qat_precision": "int4", "input_quant_bits": bits, "seed": 42},
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    save_path = OUT_DIR / run_id / "result.json"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    int4_records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(int4_records)} INT4 runs complete.")

  in8b: exists, skipping
  in4b: exists, skipping

--- QAT INT4, input_bits=2 ---
Inserted 107 quantizers


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 0.4 seconds
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 90.00% | Top-5: 90.00% | Infer: 8.13 ms/batch
  Batch [50/5000] Top-1: 95.00% | Top-5: 95.00% | Infer: 8.82 ms/batch
  Batch [60/5000] Top-1: 93.33% | Top-5: 96.67% | Infer: 9.42 ms/batch
  Batch [70/5000] Top-1: 92.50% | Top-5: 95.00% | Infer: 9.28 ms/batch
  Batch [80/5000] Top-1: 92.00% | Top-5: 96.00% | Infer: 9.10 ms/batch
  Batch [90/5000] Top-1: 93.33% | Top-5: 96.67% | Infer: 9.37 ms/batch
  Batch [100/5000] Top-1: 91.43% | Top-5: 97.14% | Infer: 9.57 ms/batch
  Batch [110/5000] Top-1: 88.75% | Top-5: 97.50% | Infer: 9.62 ms/batch
  Batch [120/5000] Top-1: 87.78% | Top-5: 97.78% | Infer: 9.89 ms/batch
  Batch [130/5000] Top-1: 87.00% | Top-5: 98.00% | Infer: 9.93 ms/batch
  Batch [140/5000] Top-1:

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 90.00% | Top-5: 90.00% | Infer: 10.26 ms/batch
  Batch [50/5000] Top-1: 95.00% | Top-5: 95.00% | Infer: 10.41 ms/batch
  Batch [60/5000] Top-1: 86.67% | Top-5: 96.67% | Infer: 10.38 ms/batch
  Batch [70/5000] Top-1: 82.50% | Top-5: 95.00% | Infer: 10.69 ms/batch
  Batch [80/5000] Top-1: 80.00% | Top-5: 94.00% | Infer: 10.54 ms/batch
  Batch [90/5000] Top-1: 80.00% | Top-5: 95.00% | Infer: 10.19 ms/batch
  Batch [100/5000] Top-1: 80.00% | Top-5: 94.29% | Infer: 10.21 ms/batch
  Batch [110/5000] Top-1: 77.50% | Top-5: 95.00% | Infer: 10.10 ms/batch
  Batch [120/5000] Top-1: 75.56% | Top-5: 95.56% | Infer: 10.07 ms/batch
  Batch [130/5000] Top-1: 75.00% | Top-5: 96.00% | Infer: 10.09 ms/batch
  Batch [140/5000] Top-1: 74.55% | Top-5: 96.36% | Infer: 10.04 ms/batch
  Batch [150/5000] Top-1: 71.67

## Results Summary

In [12]:
rows = []
for rec in int8_records + int4_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    prec = extra.get("qat_precision", "int8")
    rows.append({
        "qat_precision": f"qat_{prec}",
        "input_bits": bits,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "throughput": r.get("throughput_infer_sps", None),
    })

df = pd.DataFrame(rows).sort_values(
    ["qat_precision", "input_bits"], ascending=[True, True]
).reset_index(drop=True)
df

,qat_precision,input_bits,top1,top5,lat_ms,throughput
0,qat_int4,1,68.873,89.215,10.017,99.831
1,qat_int4,2,75.915,92.455,9.993,100.073
2,qat_int4,4,77.445,93.119,9.394,106.446
3,qat_int4,8,77.103,93.099,9.398,106.406
4,qat_int8,1,68.813,89.396,8.134,122.941
5,qat_int8,2,76.419,93.119,8.052,124.192
6,qat_int8,4,78.773,93.461,8.206,121.861
7,qat_int8,8,78.008,93.380,8.505,117.572


In [9]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_test_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/resultsv2/test_runs/qat_test_results.json
